# Features and Targets 

This notebook builds conversation-level features and target variables from the English-only ShareGPT subset created in Notebook 1.

The goal is to transform tree-structured dialogue data into a structured analytical dataset for downstream modeling.  
Each conversation is mapped to exactly one row.

Pipeline Summary

The workflow consists of the following stages:

- Data Loading  
The filtered English-only dataset is loaded from the processed data directory.

- Feature Extraction  
Conversation-level features are created from the first prompt-response pair, dialogue structure, token usage, prompt design, task cues, and orthographic quality.

- Prompt Validation  
Broken, empty, or noisy prompts are removed before representation learning.

- Embedding  
Prompt embeddings are generated using a sentence-transformer model.

- Topic Modelling  
Semantic prompt topics are inferred using BERTopic.

- Target Construction  
Proxy target variables are derived for downstream efficiency and quality modeling.

- Export  
The full dataframe, feature matrix, target table, and embeddings are saved for later notebooks.


### Design Rationale

This notebook prioritizes:

- Clear separation of feature logic and target logic
- Reproducibility through deterministic preprocessing steps
- Modular helper functions
- Consistency with Notebook 1
- Readability for later model interpretation

In [1]:
# ----------------------------
# IMPORTS + CONFIG
# ----------------------------

import json
import re
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import tiktoken
from spellchecker import SpellChecker
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_distances
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

pd.set_option("display.max_colwidth", None)


PROJECT_ROOT = Path.cwd().parent

CONFIG = {
    "processed_dir": PROJECT_ROOT / "01_data" / "02_processed",
    "features_dir": PROJECT_ROOT / "01_data" / "03_features",
    "input_file": "llm_sustainability_raw_v1-0-0.json",
    "full_dataframe_file": "llm_sustainability_strict_v1-1-0_dataframe.csv",
    "embeddings_file": "llm_sustainability_strict_v1-1-0_embeddings.npy",
    "embedding_model": "all-MiniLM-L6-v2",
    "embedding_chunk_size": 5000,
    "embedding_batch_size": 64,
    "min_prompt_length": 10,
    "random_state": 42,
}

INPUT_PATH = CONFIG["processed_dir"] / CONFIG["input_file"]
FULL_DF_PATH = CONFIG["features_dir"] / CONFIG["full_dataframe_file"]
EMBEDDINGS_PATH = CONFIG["features_dir"] / CONFIG["embeddings_file"]

CONFIG["features_dir"].mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_PATH:", INPUT_PATH)
print("Exists:", INPUT_PATH.exists())


# ----------------------------
# GLOBAL OBJECTS
# ----------------------------

WORD_RE = re.compile(r"[A-Za-z']+")
ENCODING = tiktoken.get_encoding("cl100k_base")
SPELL = SpellChecker(language="en")

c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis
INPUT_PATH: c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\02_processed\llm_sustainability_raw_v1-0-0.json
Exists: True


In [2]:
def load_json(path):
    with Path(path).open("r", encoding="utf-8") as file:
        return json.load(file)


def save_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

In [3]:
# ----------------------------
# CORE EXTRACTION
# ----------------------------

def get_first_user_prompt(tree):
    conversations = tree.get("conversations", [])

    for message in conversations:
        if message.get("from") == "human":
            return message.get("value", "")

    return ""


def get_first_prompt_response_pair(tree):
    conversations = tree.get("conversations", [])

    for i in range(len(conversations) - 1):
        current = conversations[i]
        next_msg = conversations[i + 1]

        if current.get("from") == "human" and next_msg.get("from") == "gpt":
            return current.get("value", ""), next_msg.get("value", "")

    return "", ""


def get_words(text):
    return WORD_RE.findall(str(text).lower())

In [4]:
# ----------------------------
# TOKEN AND STRUCTURE FEATURES
# ----------------------------

def count_tokens(text):
    return len(
        ENCODING.encode(
            str(text),
            disallowed_special=()
        )
    )


def count_user_prompts(tree):
    return sum(
        message.get("from") == "human"
        for message in tree.get("conversations", [])
    )


def count_total_user_tokens(tree):
    total = 0

    for message in tree.get("conversations", []):
        if message.get("from") == "human":
            total += count_tokens(message.get("value", ""))

    return total


def count_total_assistant_tokens(tree):
    total = 0

    for message in tree.get("conversations", []):
        if message.get("from") == "gpt":
            total += count_tokens(message.get("value", ""))

    return total

In [5]:
# ----------------------------
# PROMPT DESIGN FEATURES
# ----------------------------

def has_role_instruction(prompt):
    prompt = str(prompt).lower()

    role_patterns = [
        "act as",
        "you are",
        "you're",
        "pretend you are",
        "imagine you are",
        "take the role",
        "role of",
    ]

    return any(pattern in prompt for pattern in role_patterns)


def has_audience_or_level_instruction(prompt):
    prompt = str(prompt).lower()

    patterns = [
        "explain it to me like",
        "explain like i am",
        "explain like i'm",
        "eli5",
        "like i am 5",
        "like i'm 5",
        "like i am 10",
        "like i'm 10",
        "for beginners",
        "to a beginner",
        "in simple terms",
        "plain english",
        "non-technical",
        "for a child",
        "for a high school student",
    ]

    return any(pattern in prompt for pattern in patterns)


def has_format_instruction(prompt):
    prompt = str(prompt).lower()

    format_keywords = [
        "bullet point",
        "bullet points",
        "table",
        "json",
        "csv",
        "list",
        "markdown",
        "format",
        "section",
        "sections",
        "outline",
        "step by step",
        "numbered",
    ]

    return any(keyword in prompt for keyword in format_keywords)


def count_questions(prompt):
    return str(prompt).count("?")


def prompt_style(prompt):
    prompt_lower = str(prompt).lower().strip()

    instruction_verbs = [
        "write", "summarize", "explain", "create", "generate", "translate",
        "list", "compare", "analyze", "give", "make", "draft", "compose",
        "rewrite", "classify", "extract", "find", "calculate",
    ]

    if "?" in prompt_lower:
        return "question"

    if any(prompt_lower.startswith(verb) for verb in instruction_verbs):
        return "instruction"

    return "other"

In [ ]:
# ----------------------------
# TEXT QUALITY AND TASK TYPE
# ----------------------------

def orthographic_error_rate(prompt):                         # questionable feature for mixed texts like chats, social media
    words = get_words(prompt)

    if not words:
        return 0.0

    unknown_words = SPELL.unknown(words)

    return len(unknown_words) / len(words)


def classify_task_type(prompt):
    prompt = str(prompt).lower()

    if any(word in prompt for word in [
        "python", "java", "javascript", "typescript", "sql", "html", "css",
        "api", "debug", "error", "exception", "bash", "terminal",
        "command", "linux", "c++", "cpp"
    ]):
        return "coding"

    if any(word in prompt for word in [
        "email", "subject line", "reply to", "newsletter"
    ]):
        return "email_writing"

    if any(word in prompt for word in [
        "summarize", "summary", "tl;dr", "key points"
    ]):
        return "summarization"

    if any(word in prompt for word in [
        "translate", "translation", "in english", "into english",
        "into spanish", "into french"
    ]):
        return "translation"

    if any(word in prompt for word in [
        "explain", "what is", "how does", "why does",
        "difference between", "simple terms"
    ]):
        return "explanation"

    if any(word in prompt for word in [
        "write", "draft", "compose", "story", "poem",
        "article", "script", "blog post"
    ]):
        return "writing_generation"

    if any(word in prompt for word in [
        "idea", "ideas", "brainstorm", "suggest", "recommend"
    ]):
        return "brainstorming"

    if any(word in prompt for word in [
        "act as", "pretend", "roleplay", "you are now", "simulate"
    ]):
        return "roleplay"

    return "general_assistance"

In [7]:
# ----------------------------
# PROMPT VALIDATION
# ----------------------------

def is_valid_prompt(text):
    text = str(text)

    if not text or len(text) < CONFIG["min_prompt_length"]:
        return False

    noise_markers = [
        "root@",
        "-----",
        "Enter pass phrase",
        "Select an option",
    ]

    return not any(marker in text for marker in noise_markers)

In [8]:
# ----------------------------
# CONVERSATION TO ROW
# ----------------------------

def build_conversation_row(tree):
    first_prompt, first_response = get_first_prompt_response_pair(tree)

    total_turns = len(tree.get("conversations", []))
    total_user_prompts = count_user_prompts(tree)

    total_user_tokens = count_total_user_tokens(tree)
    total_assistant_tokens = count_total_assistant_tokens(tree)
    total_tokens = total_user_tokens + total_assistant_tokens

    follow_up_prompts = max(0, total_user_prompts - 1)
    needs_follow_up = int(follow_up_prompts > 0)

    return {
        "conversation_id": tree.get("id"),

        "first_prompt": first_prompt,
        "first_response": first_response,

        "first_prompt_tokens": count_tokens(first_prompt),
        "first_response_tokens": count_tokens(first_response),

        "total_turns": total_turns,
        "interaction_rounds": total_turns / 2,
        "total_user_prompts": total_user_prompts,
        "follow_up_prompts": follow_up_prompts,
        "needs_follow_up": needs_follow_up,

        "total_user_tokens": total_user_tokens,
        "total_assistant_tokens": total_assistant_tokens,
        "total_tokens": total_tokens,
        "log_total_tokens": np.log1p(total_tokens),

        "has_role_instruction": int(has_role_instruction(first_prompt)),
        "has_audience_or_level_instruction": int(has_audience_or_level_instruction(first_prompt)),
        "has_format_instruction": int(has_format_instruction(first_prompt)),
        "question_count": int(count_questions(first_prompt)),
        "prompt_style": prompt_style(first_prompt),

        "task_type": classify_task_type(first_prompt),
        "orthographic_error_rate": orthographic_error_rate(first_prompt),
    }

In [9]:
# ----------------------------
# BUILD DATAFRAME
# ----------------------------

def build_dataframe(data):
    rows = [build_conversation_row(tree) for tree in data]
    return pd.DataFrame(rows)

In [10]:
# ----------------------------
# EMBEDDING
# ----------------------------

def embed_chunked(texts, model, chunk_size=5000, batch_size=64):
    all_embeddings = []

    for i in tqdm(range(0, len(texts), chunk_size)):
        chunk = texts[i:i + chunk_size]

        emb = model.encode(
            chunk,
            batch_size=batch_size,
            show_progress_bar=False
        )

        all_embeddings.append(emb)

    return np.concatenate(all_embeddings, axis=0)

In [11]:
# ----------------------------
# LOAD DATA
# ----------------------------

english_data = load_json(INPUT_PATH)

print(f"Loaded conversations: {len(english_data)}")

# ----------------------------
# CHECK CONVERSATION IDS
# ----------------------------

conversation_ids = [
    item.get("id")
    for item in english_data
    if isinstance(item, dict) and item.get("id") is not None
]

print(f"Conversation IDs: {len(conversation_ids)}")
print(f"Unique Conversation IDs: {len(set(conversation_ids))}")
print(f"Duplicates: {len(conversation_ids) - len(set(conversation_ids))}")

# ----------------------------
# BUILD FEATURES
# ----------------------------

start = perf_counter()

df = build_dataframe(english_data)

duration = perf_counter() - start

print(f"Rows built: {len(df)}")
print(f"Time: {duration:.2f}s")
print(f"Rows/sec: {len(df)/duration:.0f}")

Loaded conversations: 57254
Conversation IDs: 57254
Unique Conversation IDs: 57254
Duplicates: 0
Rows built: 57254
Time: 151.80s
Rows/sec: 377


In [12]:
# ----------------------------
# FILTER INVALID PROMPTS
# ----------------------------

before = len(df)

df["is_valid_prompt"] = df["first_prompt"].apply(is_valid_prompt).astype(int)

df = df[
    df["first_prompt"].fillna("").str.len().gt(0) &
    df["first_response"].fillna("").str.len().gt(0) &
    df["is_valid_prompt"].eq(1)
].reset_index(drop=True)

duration = perf_counter() - start

print(f"Before filter: {before}")
print(f"After filter: {len(df)}")
print(f"Removed: {before - len(df)}")
print(f"Time: {duration:.2f}s")


Before filter: 57254
After filter: 55659
Removed: 1595
Time: 152.10s


In [13]:
# ----------------------------
# RUN EMBEDDING
# ----------------------------

embedder = SentenceTransformer(CONFIG["embedding_model"])

prompts = df["first_prompt"].fillna("").astype(str).to_numpy()

X_emb = embed_chunked(
    prompts,
    model=embedder,
    chunk_size=CONFIG["embedding_chunk_size"],
    batch_size=CONFIG["embedding_batch_size"]
)

assert len(X_emb) == len(df)

np.save(EMBEDDINGS_PATH, X_emb)

print("Embeddings shape:", X_emb.shape)

100%|██████████| 12/12 [12:05<00:00, 60.46s/it]

Embeddings shape: (55659, 384)


In [14]:
# ----------------------------
# TOPIC MODELLING
# ----------------------------

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5
)

umap_model = UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=CONFIG["random_state"]
)

hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric="euclidean",
    prediction_data=True
)

topic_model = BERTopic(
    embedding_model=embedder,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    min_topic_size=50,
    nr_topics="auto"
)

topics, probs = topic_model.fit_transform(df["first_prompt"].tolist())

df["topic"] = topics
df["topic_prob"] = probs.max(axis=1)

KeyboardInterrupt: 

In [ ]:
# df["topic"].value_counts(normalize=True)
df["topic"].value_counts()

topic
-1     25822
 0      9623
 1       546
 2       531
 3       493
       ...  
 66       52
 67       52
 68       51
 69       51
 70       50
Name: count, Length: 72, dtype: int64

In [ ]:
def inspect_topic(topic_model, topic_id, docs, top_n=10, sample_n=5):
    print(f"Topic {topic_id}")
    print("-" * 80)

    words = topic_model.get_topic(topic_id)
    if words:
        print("Top words:")
        for word, weight in words[:top_n]:
            print(f"  {word:20s} {weight:.4f}")
    else:
        print("No words found for this topic.")

    print("\nExample documents:")
    topic_docs = [doc for doc, topic in zip(docs, topic_model.topics_) if topic == topic_id]
    for i, doc in enumerate(topic_docs[:sample_n], start=1):
        print(f"\n[{i}] {doc[:500]}")

In [ ]:
topic_labels = topic_model.generate_topic_labels(nr_words=5)

label_map = {
    topic_id: label
    for topic_id, label in zip(sorted(topic_model.get_topics().keys()), topic_labels)
}

df["topic_label"] = df["topic"].map(label_map)


topic_distribution = (
    df.groupby(["topic", "topic_label"])
      .size()
      .reset_index(name="count")
      .sort_values(["count", "topic"], ascending=[False, True])
)

topic_distribution.head(20)

,topic,topic_label,count
0,-1,-1_like_use_data_write_time,25822
1,0,0_dan_chatgpt_prompt_content_developer,9623
2,1,1_energy_electric_battery_norway_agriculture,546
3,2,2_const_react_import_div_gt,531
4,3,3_tax_market_financial_bitcoin_stock,493
5,4,4_trip_visit_travel_day_places,448
6,5,5_fa_patient_cells_drug_ct,441
7,6,6_000_minutes_total_hours_did,424
8,7,7_gt_null_select_gt gt_id,413
9,8,8_piano_music_song_album_left hand,372


In [ ]:
# ----------------------------
# EMBEDDING NOVELTY
# ----------------------------

centroid = X_emb.mean(axis=0)

df["embedding_novelty"] = cosine_distances(
    X_emb,
    centroid.reshape(1, -1)
).flatten()

In [ ]:
# ----------------------------
# TARGETS
# ----------------------------

df["target_cost"] = df["first_response_tokens"]

refusal_pattern = r"(?i)(i can't|i cannot|i’m sorry|i am sorry|as an ai|i do not have access)"

df["target_success_refusal"] = (
    ~df["first_response"].fillna("").str.contains(refusal_pattern, regex=True)
).astype(int)

clarification_pattern = r"(?i)(could you clarify|do you mean|can you specify|i'm not sure|i may have misunderstood|did you mean)"
question_pattern = r"\?"

df["target_needs_clarification"] = (
    df["first_response"].fillna("").str.contains(clarification_pattern, regex=True)
    | (
        df["first_response"].fillna("").str.contains(question_pattern, regex=True)
        & df["first_response"].fillna("").str.len().gt(20)
    )
).astype(int)

pd.crosstab(
    df["target_needs_clarification"],
    df["target_success_refusal"],
    rownames=["needs_clarification"],
    colnames=["success_refusal"],
    margins=True
)

success_refusal,0,1,All
needs_clarification,,,
0,1570,38134,39704
1,341,6388,6729
All,1911,44522,46433


In [ ]:
print(df.columns.tolist())

['conversation_id', 'first_prompt', 'first_response', 'first_prompt_tokens', 'first_response_tokens', 'total_turns', 'interaction_rounds', 'total_user_prompts', 'follow_up_prompts', 'needs_follow_up', 'total_user_tokens', 'total_assistant_tokens', 'total_tokens', 'log_total_tokens', 'has_role_instruction', 'has_audience_or_level_instruction', 'has_format_instruction', 'question_count', 'prompt_style', 'task_type', 'orthographic_error_rate', 'is_valid_prompt', 'topic', 'topic_prob', 'topic_label', 'embedding_novelty', 'target_cost', 'target_success_refusal', 'target_needs_clarification']


In [ ]:
# ----------------------------
# SAVE
# ----------------------------

save_csv(df, FULL_DF_PATH)

print(f"Saved full dataframe: {FULL_DF_PATH}")
print(f"Saved embeddings: {EMBEDDINGS_PATH}")

Saved full dataframe: c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\03_features\llm_sustainability_strict_v1-1-0_dataframe.csv
Saved embeddings: c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\03_features\llm_sustainability_strict_v1-1-0_embeddings.npy


In [ ]:
# ----------------------------
# DIAGNOSTICS
# ----------------------------

print("Full dataframe shape:", df.shape)
print()
print(df["target_success_refusal"].value_counts(dropna=False))
print(df["target_needs_clarification"].value_counts(dropna=False))
print()
print(df["target_cost"].describe())
print()
print(df[["first_prompt", "task_type", "prompt_style", "topic_label", "target_success_refusal"]].head(10))
print(df[["first_prompt", "task_type", "prompt_style", "topic_label", "target_needs_clarification"]].head(10))

Full dataframe shape: (46433, 29)

target_success_refusal
1    44522
0     1911
Name: count, dtype: int64
target_needs_clarification
0    39704
1     6729
Name: count, dtype: int64

count    46433.00000
mean       498.11649
std        925.61692
min          1.00000
25%        161.00000
50%        337.00000
75%        583.00000
max      76828.00000
Name: target_cost, dtype: float64

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       